# API Anomaly Detection




In [ ]:

import json
import math
import pickle
import random
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional
from urllib.parse import urlparse, parse_qs, unquote

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Paths for Dataset 3
TRAIN_DATASET_PATH = Path("datasets/dataset_3_train.json")
if not TRAIN_DATASET_PATH.exists():
    TRAIN_DATASET_PATH = Path("../datasets/dataset_3_train.json")

VALIDATION_DATASET_PATH = Path("datasets/dataset_3_val.json")
if not VALIDATION_DATASET_PATH.exists():
    VALIDATION_DATASET_PATH = Path("../datasets/dataset_3_val.json")

STORAGE_DIR = Path("storage")
STORAGE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = STORAGE_DIR / "isolation_forest.pkl"
FEATURE_CONFIG_PATH = STORAGE_DIR / "feature_config.json"

print(f"Train dataset path: {TRAIN_DATASET_PATH}")
print(f"Validation dataset path: {VALIDATION_DATASET_PATH}")
print(f"Model output path: {MODEL_PATH}")
print(f"Feature config output path: {FEATURE_CONFIG_PATH}")


Train dataset path: datasets/dataset_3_train.json
Validation dataset path: datasets/dataset_3_val.json
Model output path: storage/isolation_forest.pkl
Feature config output path: storage/feature_config.json


In [3]:
# ============================================================
# 2. Dataset Loading
# ============================================================

def load_dataset(path: Path | str) -> list[dict]:
    with open(path, "r", encoding="utf-8") as file:
        records = json.load(file)
    if not isinstance(records, list):
        raise ValueError("Dataset must be a JSON list of request/response records.")
    return records

train_records = load_dataset(TRAIN_DATASET_PATH)
pure_val_records = load_dataset(VALIDATION_DATASET_PATH)

print("Total training records loaded:", len(train_records))
print("Total pure validation records loaded:", len(pure_val_records))


Total training records loaded: 144500
Total pure validation records loaded: 25500


In [4]:
# ============================================================
# 3. Request Parser
# ============================================================

@dataclass
class ApiRequest:
    method: str
    url: str
    endpoint: str
    query_string: str
    headers: Dict[str, Any]
    body: str
    attack_tag: Optional[str]
    status_code: int
    response_status: str
    response_headers: Dict[str, Any]
    response_body: str
    raw_request: Dict[str, Any]

class RequestParser:
    def normalize_record(self, record: Dict[str, Any]) -> ApiRequest:
        request = record.get("request", {}) or {}
        response = record.get("response", {}) or {}
        url = str(request.get("url", "") or "")
        parsed_url = urlparse(url)
        return ApiRequest(
            method=str(request.get("method", "GET") or "GET").upper(),
            url=url,
            endpoint=parsed_url.path or "/",
            query_string=parsed_url.query or "",
            headers=request.get("headers", {}) or {},
            body=str(request.get("body", "") or ""),
            attack_tag=request.get("Attack_Tag"),
            status_code=int(response.get("status_code", 0) or 0),
            response_status=str(response.get("status", "") or ""),
            response_headers=response.get("headers", {}) or {},
            response_body=str(response.get("body", "") or ""),
            raw_request=record,
        )

    def parse_dataset(self, records: list[dict]) -> list[ApiRequest]:
        return [self.normalize_record(record) for record in records]


In [5]:
# ============================================================
# 4. Feature Extractor
# ============================================================

@dataclass
class ExtractedFeatures:
    url_length: int
    endpoint_depth: int
    query_param_count: int
    body_length: int
    header_count: int
    cookie_length: int
    user_agent_length: int
    special_char_count: int
    special_char_density: float
    sql_pattern_score: float
    xss_pattern_score: float
    traversal_pattern_score: float
    command_pattern_score: float
    log_injection_pattern_score: float
    log4j_pattern_score: float
    cookie_injection_pattern_score: float
    composite_anomaly_score: float
    response_body_length: int
    response_header_count: int
    status_code: int
    is_error_status: int

class FeatureExtractor:
    SQL_PATTERNS = [
        re.compile(r"(\b(union\s+select|select\s+.*\s+from|insert\s+into|delete\s+from|drop\s+table|information_schema|benchmark\s*\(|sleep\s*\(|from\s+employees|limit\s+\d+|where\s+|or\s+|and\s+)\b|\x27\s*or\s*\x27|\x27\s*or\s*1\s*=\s*1|(\s--|--\s|;\s*--|\x27--|\"--|--$)|;\s*select)", re.I),
    ]

    XSS_PATTERNS = [
        re.compile(r"(<script.*?>|alert\s*\(|onerror\s*=|onload\s*=|document\.cookie|javascript:|<iframe>|<svg|<body|img\s+src=)", re.I),
    ]

    TRAVERSAL_PATTERNS = [
        re.compile(r"(\.\./|\.\.\\|%2e%2e%2f|%2e%2e/|\.\.%2f|windows\.ini|/etc/passwd|win\.ini|boot\.ini)", re.I),
    ]

    COMMAND_PATTERNS = [
        re.compile(r"(\b(whoami|powershell|os\.system|builtins|subprocess|cmd\.exe|cat\s+/etc)\b)", re.I),
    ]

    LOG_INJECTION_PATTERNS = [
        re.compile(r"(%0a|%0d|\\n|\\r|\n|\r).*(signin:|error:|admin|user)", re.I),
    ]

    LOG4J_PATTERNS = [
        re.compile(r"(\$\{jndi:(ldap|rmi|dns)|jndi:(ldap|rmi|dns))", re.I),
    ]

    COOKIE_INJECTION_PATTERNS = [
        re.compile(r"(gASV[A-Za-z0-9+/=]{10,}|builtins|__main__|pickle|namedtuple|cposix|eval\(|exec\(|import\s+os|powershell)", re.I),
    ]

    SPECIAL_CHARS = ["'", '"', "<", ">", "=", ";", "(", ")", "{", "}", "[", "]", "$", "|", "&", "\\", "/", ":", "%"]

    def extract(self, req: ApiRequest) -> ExtractedFeatures:
        dec_url = unquote(req.url)
        dec_query = unquote(req.query_string)
        dec_headers = unquote(str(req.headers))
        cookie_text = str(req.headers.get("Cookie", "")) + " " + str(req.headers.get("Set-Cookie", ""))
        dec_cookie = unquote(cookie_text)
        ua_str = str(req.headers.get("User-Agent", ""))

        combined_text = " ".join([dec_url, req.endpoint, dec_query, dec_headers, req.body, req.response_body]).lower()
        url_body_text = (dec_url + " " + req.body).lower()

        spec_cnt = sum(combined_text.count(c) for c in self.SPECIAL_CHARS)
        total_len = len(req.url) + len(req.body) + len(cookie_text) + 1
        spec_density = float(spec_cnt) / float(total_len)

        sql_hit = 1.0 if any(p.search(url_body_text) for p in self.SQL_PATTERNS) else 0.0
        xss_hit = 1.0 if any(p.search(url_body_text) for p in self.XSS_PATTERNS) else 0.0
        trav_hit = 1.0 if any(p.search(url_body_text) for p in self.TRAVERSAL_PATTERNS) else 0.0
        cmd_hit = 1.0 if any(p.search(combined_text) for p in self.COMMAND_PATTERNS) else 0.0
        log_hit = 1.0 if any(p.search(url_body_text) for p in self.LOG_INJECTION_PATTERNS) else 0.0
        log4j_hit = 1.0 if any(p.search(dec_headers) for p in self.LOG4J_PATTERNS) else 0.0
        cookie_hit = 1.0 if any(p.search(dec_cookie) for p in self.COOKIE_INJECTION_PATTERNS) else 0.0

        composite = sql_hit + xss_hit + trav_hit + cmd_hit + log_hit + log4j_hit + cookie_hit

        return ExtractedFeatures(
            url_length=len(req.url),
            endpoint_depth=len([p for p in req.endpoint.split("/") if p]),
            query_param_count=len(parse_qs(req.query_string)),
            body_length=len(req.body),
            header_count=len(req.headers),
            cookie_length=len(cookie_text),
            user_agent_length=len(ua_str),
            special_char_count=spec_cnt,
            special_char_density=spec_density,
            sql_pattern_score=sql_hit,
            xss_pattern_score=xss_hit,
            traversal_pattern_score=trav_hit,
            command_pattern_score=cmd_hit,
            log_injection_pattern_score=log_hit,
            log4j_pattern_score=log4j_hit,
            cookie_injection_pattern_score=cookie_hit,
            composite_anomaly_score=composite,
            response_body_length=len(req.response_body),
            response_header_count=len(req.response_headers),
            status_code=req.status_code,
            is_error_status=1 if req.status_code >= 400 else 0,
        )

    def to_vector(self, f: ExtractedFeatures) -> list[float]:
        return [
            float(f.url_length),
            float(f.endpoint_depth),
            float(f.query_param_count),
            float(f.body_length),
            float(f.header_count),
            float(f.cookie_length),
            float(f.user_agent_length),
            float(f.special_char_count),
            float(f.special_char_density),
            float(f.sql_pattern_score),
            float(f.xss_pattern_score),
            float(f.traversal_pattern_score),
            float(f.command_pattern_score),
            float(f.log_injection_pattern_score),
            float(f.log4j_pattern_score),
            float(f.cookie_injection_pattern_score),
            float(f.composite_anomaly_score),
            float(f.response_body_length),
            float(f.response_header_count),
            float(f.status_code),
            float(f.is_error_status),
        ]

FEATURE_NAMES = [
    "url_length",
    "endpoint_depth",
    "query_param_count",
    "body_length",
    "header_count",
    "cookie_length",
    "user_agent_length",
    "special_char_count",
    "special_char_density",
    "sql_pattern_score",
    "xss_pattern_score",
    "traversal_pattern_score",
    "command_pattern_score",
    "log_injection_pattern_score",
    "log4j_pattern_score",
    "cookie_injection_pattern_score",
    "composite_anomaly_score",
    "response_body_length",
    "response_header_count",
    "status_code",
    "is_error_status",
]

print(f"Total features configured: {len(FEATURE_NAMES)}")


Total features configured: 21


In [ ]:

def prepare_data(records: list[dict]) -> tuple[np.ndarray, np.ndarray, list[ApiRequest], pd.DataFrame]:
    parser = RequestParser()
    extractor = FeatureExtractor()

    parsed_requests = parser.parse_dataset(records)
    X = []
    y = []
    rows = []

    for req in parsed_requests:
        features = extractor.extract(req)
        vector = extractor.to_vector(features)
        label = 1 if req.attack_tag else 0

        X.append(vector)
        y.append(label)

        row = asdict(features)
        row["label"] = label
        row["attack_tag"] = req.attack_tag if req.attack_tag else "Benign"
        row["method"] = req.method
        row["endpoint"] = req.endpoint
        rows.append(row)

    return (
        np.array(X, dtype=float),
        np.array(y, dtype=int),
        parsed_requests,
        pd.DataFrame(rows),
    )

X_all, y_all, parsed_all_requests, features_df = prepare_data(train_records)

# Stratified 70% Train, 15% Validation, 15% Test partition
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.30, random_state=RANDOM_STATE, stratify=y_all
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"X_train shape: {X_train.shape}, Benign: {(y_train == 0).sum()}, Attack: {(y_train == 1).sum()}")
print(f"X_val shape:   {X_val.shape}, Benign: {(y_val == 0).sum()}, Attack: {(y_val == 1).sum()}")
print(f"X_test shape:  {X_test.shape}, Benign: {(y_test == 0).sum()}, Attack: {(y_test == 1).sum()}")


X_train shape: (101150, 21), Benign: 89869, Attack: 11281
X_val shape:   (21675, 21), Benign: 19257, Attack: 2418
X_test shape:  (21675, 21), Benign: 19258, Attack: 2417


In [ ]:

def minmax_fit_transform(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    min_values = X.min(axis=0)
    max_values = X.max(axis=0)
    denominator = max_values - min_values
    denominator[denominator == 0] = 1.0
    X_scaled = (X - min_values) / denominator
    return X_scaled, min_values, max_values

def minmax_transform(X: np.ndarray, min_values: np.ndarray, max_values: np.ndarray) -> np.ndarray:
    denominator = max_values - min_values
    denominator[denominator == 0] = 1.0
    return (X - min_values) / denominator

X_train_scaled, min_values, max_values = minmax_fit_transform(X_train)
X_val_scaled = minmax_transform(X_val, min_values, max_values)
X_test_scaled = minmax_transform(X_test, min_values, max_values)

print("Min-Max scaling completed.")


Min-Max scaling completed.


In [ ]:

@dataclass
class IsolationTreeNode:
    feature_index: Optional[int] = None
    split_value: Optional[float] = None
    left: Optional["IsolationTreeNode"] = None
    right: Optional["IsolationTreeNode"] = None
    size: int = 0
    depth: int = 0
    is_leaf: bool = False

def _node_to_dict(node: Optional[IsolationTreeNode]) -> Optional[dict]:
    if node is None:
        return None
    return {
        "feature_index": node.feature_index,
        "split_value": node.split_value,
        "size": node.size,
        "depth": node.depth,
        "is_leaf": node.is_leaf,
        "left": _node_to_dict(node.left),
        "right": _node_to_dict(node.right),
    }

def _node_from_dict(d: Optional[dict]) -> Optional[IsolationTreeNode]:
    if d is None:
        return None
    return IsolationTreeNode(
        feature_index=d.get("feature_index"),
        split_value=d.get("split_value"),
        size=d.get("size", 0),
        depth=d.get("depth", 0),
        is_leaf=d.get("is_leaf", False),
        left=_node_from_dict(d.get("left")),
        right=_node_from_dict(d.get("right")),
    )

def average_path_length(n: int) -> float:
    if n <= 1:
        return 0.0
    if n == 2:
        return 1.0
    euler_constant = 0.5772156649
    return 2.0 * (math.log(n - 1) + euler_constant) - (2.0 * (n - 1) / n)

class IsolationTree:
    def __init__(self, max_depth: int):
        self.max_depth = max_depth
        self.root: Optional[IsolationTreeNode] = None

    def fit(self, X: np.ndarray, feature_weights: Optional[np.ndarray] = None):
        self.root = self._build_tree(X, depth=0, feature_weights=feature_weights)
        return self

    def _build_tree(self, X: np.ndarray, depth: int, feature_weights: Optional[np.ndarray] = None) -> IsolationTreeNode:
        node = IsolationTreeNode(size=len(X), depth=depth)
        if depth >= self.max_depth or len(X) <= 1:
            node.is_leaf = True
            return node

        mins = np.min(X, axis=0)
        maxs = np.max(X, axis=0)
        valid_features = np.where(maxs > mins)[0]
        if len(valid_features) == 0:
            node.is_leaf = True
            return node

        if feature_weights is not None:
            w = feature_weights[valid_features]
            w_sum = w.sum()
            p = w / w_sum if w_sum > 0 else None
        else:
            p = None

        feature_index = int(np.random.choice(valid_features, p=p))
        split_value = float(np.random.uniform(mins[feature_index], maxs[feature_index]))

        left_mask = X[:, feature_index] < split_value
        right_mask = ~left_mask

        node.feature_index = feature_index
        node.split_value = split_value
        node.left = self._build_tree(X[left_mask], depth + 1, feature_weights)
        node.right = self._build_tree(X[right_mask], depth + 1, feature_weights)
        return node

    def path_length(self, x: np.ndarray) -> float:
        return self._path_length(x, self.root, depth=0)

    def _path_length(self, x: np.ndarray, node: Optional[IsolationTreeNode], depth: int) -> float:
        if node is None:
            return float(depth)
        if node.is_leaf:
            return float(depth) + average_path_length(node.size)
        if x[node.feature_index] < node.split_value:
            return self._path_length(x, node.left, depth + 1)
        return self._path_length(x, node.right, depth + 1)

    def path_lengths_batch(self, X: np.ndarray) -> np.ndarray:
        out = np.zeros(len(X), dtype=float)
        self._eval_batch(X, np.arange(len(X)), self.root, 0, out)
        return out

    def _eval_batch(self, X: np.ndarray, indices: np.ndarray, node: Optional[IsolationTreeNode], depth: int, out: np.ndarray):
        if len(indices) == 0:
            return
        if node is None or node.is_leaf:
            c = average_path_length(node.size if node else 0)
            out[indices] = float(depth) + c
            return
        vals = X[indices, node.feature_index]
        left_mask = vals < node.split_value
        self._eval_batch(X, indices[left_mask], node.left, depth + 1, out)
        self._eval_batch(X, indices[~left_mask], node.right, depth + 1, out)

class IsolationForestScratch:
    def __init__(
        self,
        n_trees: int = 300,
        sample_size: int = 512,
        contamination: float = 0.1115,
        random_state: Optional[int] = 42,
    ):
        self.n_trees = n_trees
        self.sample_size = sample_size
        self.contamination = contamination
        self.random_state = random_state
        self.trees: List[IsolationTree] = []
        self.threshold_: Optional[float] = None
        self.feature_weights_: Optional[np.ndarray] = None

        if random_state is not None:
            random.seed(random_state)
            np.random.seed(random_state)

    def fit(self, X: np.ndarray):
        X = np.asarray(X, dtype=float)
        n_samples = X.shape[0]
        actual_sample_size = min(self.sample_size, n_samples)
        max_depth = math.ceil(math.log2(actual_sample_size)) + 2

        variances = np.var(X, axis=0) + 1e-4
        weights = np.sqrt(variances)
        weights[9:17] *= 5.0
        weights = weights / weights.sum()
        self.feature_weights_ = weights

        self.trees = []
        for _ in range(self.n_trees):
            sample_indices = np.random.choice(n_samples, size=actual_sample_size, replace=False)
            tree = IsolationTree(max_depth=max_depth)
            tree.fit(X[sample_indices], feature_weights=self.feature_weights_)
            self.trees.append(tree)

        sample_eval_idx = np.random.choice(n_samples, size=min(10000, n_samples), replace=False)
        train_scores = self.anomaly_score(X[sample_eval_idx])
        self.threshold_ = float(np.percentile(train_scores, 100 * (1 - self.contamination)))
        return self

    def anomaly_score(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        normalizer = average_path_length(self.sample_size)
        if normalizer == 0:
            return np.zeros(len(X))
        all_lengths = np.zeros(len(X), dtype=float)
        for tree in self.trees:
            all_lengths += tree.path_lengths_batch(X)
        avg_path = all_lengths / len(self.trees)
        return 2.0 ** (-avg_path / normalizer)

    def predict_with_threshold(self, X: np.ndarray, threshold: float) -> np.ndarray:
        scores = self.anomaly_score(X)
        return np.where(scores >= threshold, -1, 1)

    def predict(self, X: np.ndarray) -> np.ndarray:
        if self.threshold_ is None:
            raise ValueError("Model threshold is not set. Train the model first.")
        return self.predict_with_threshold(X, self.threshold_)

    def to_dict(self) -> dict:
        return {
            "n_trees": self.n_trees,
            "sample_size": self.sample_size,
            "contamination": self.contamination,
            "random_state": self.random_state,
            "threshold": self.threshold_,
            "feature_weights": self.feature_weights_.tolist() if self.feature_weights_ is not None else None,
            "trees": [_node_to_dict(tree.root) for tree in self.trees],
        }

    @classmethod
    def from_dict(cls, data: dict) -> "IsolationForestScratch":
        model = cls(
            n_trees=data["n_trees"],
            sample_size=data["sample_size"],
            contamination=data["contamination"],
            random_state=data["random_state"],
        )
        model.threshold_ = data["threshold"]
        if data.get("feature_weights") is not None:
            model.feature_weights_ = np.array(data["feature_weights"], dtype=float)
        model.trees = []
        for tree_dict in data["trees"]:
            tree = IsolationTree(max_depth=0)
            tree.root = _node_from_dict(tree_dict)
            model.trees.append(tree)
        return model

    def save(self, path: str | Path):
        with open(path, "wb") as file:
            pickle.dump(self.to_dict(), file)

    @classmethod
    def load(cls, path: str | Path) -> "IsolationForestScratch":
        with open(path, "rb") as file:
            data = pickle.load(file)
        if isinstance(data, dict):
            return cls.from_dict(data)
        return data


In [ ]:

print("Training self-built Isolation Forest on Dataset 3 training partition...")
model = IsolationForestScratch(
    n_trees=300,
    sample_size=512,
    contamination=0.1115,
    random_state=RANDOM_STATE,
)

model.fit(X_train_scaled)

print("Model training completed successfully.")
print("Initial threshold from contamination:", round(model.threshold_, 4))


Training self-built Isolation Forest on Dataset 3 training partition...
Model training completed successfully.
Initial threshold from contamination: 0.5301


In [ ]:

def calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }

val_scores = model.anomaly_score(X_val_scaled)
threshold_results = []

for threshold in np.linspace(val_scores.min(), val_scores.max(), 500):
    y_pred = np.where(val_scores >= threshold, 1, 0)
    metrics = calculate_metrics(y_val, y_pred)
    threshold_results.append({
        "threshold": round(float(threshold), 4),
        **metrics,
    })

threshold_df = pd.DataFrame(threshold_results)
best_row = threshold_df.sort_values(
    by=["f1_score", "accuracy", "recall"],
    ascending=False
).iloc[0]

best_threshold = float(best_row["threshold"])
print(f"Optimal tuned threshold: {best_threshold:.4f}")
print(f"Validation Accuracy:     {best_row['accuracy'] * 100:.2f}%")
print(f"Validation F1-Score:     {best_row['f1_score'] * 100:.2f}%")

model.threshold_ = best_threshold


Optimal tuned threshold: 0.5752
Validation Accuracy:     97.88%
Validation F1-Score:     89.90%


In [ ]:

# 1. Evaluate on Validation Partition
val_raw_pred = model.predict_with_threshold(X_val_scaled, best_threshold)
val_y_pred = np.where(val_raw_pred == -1, 1, 0)
val_metrics = calculate_metrics(y_val, val_y_pred)
val_roc_auc = float(roc_auc_score(y_val, val_scores))
val_metrics["roc_auc"] = val_roc_auc

# 2. Evaluate on Unseen Test Partition
test_scores = model.anomaly_score(X_test_scaled)
test_raw_pred = model.predict_with_threshold(X_test_scaled, best_threshold)
test_y_pred = np.where(test_raw_pred == -1, 1, 0)
test_metrics = calculate_metrics(y_test, test_y_pred)
test_roc_auc = float(roc_auc_score(y_test, test_scores))
test_metrics["roc_auc"] = test_roc_auc

# 3. Evaluate on Pure Validation Dataset (dataset_3_val.json)
parser = RequestParser()
extractor = FeatureExtractor()
pure_val_parsed = parser.parse_dataset(pure_val_records)
pure_val_X = np.array([extractor.to_vector(extractor.extract(r)) for r in pure_val_parsed], dtype=float)
pure_val_scaled = minmax_transform(pure_val_X, min_values, max_values)
pure_val_raw_pred = model.predict_with_threshold(pure_val_scaled, best_threshold)
pure_val_y_pred = np.where(pure_val_raw_pred == -1, 1, 0)
pure_val_accuracy = float(accuracy_score(np.zeros(len(pure_val_y_pred)), pure_val_y_pred))

print("==================== VALIDATION METRICS ====================")
print(json.dumps(val_metrics, indent=2))

print("\n==================== UNSEEN TEST SET METRICS ====================")
print(json.dumps(test_metrics, indent=2))
print(f"Test Accuracy:  {test_metrics['accuracy'] * 100:.2f}%")
print(f"Test Precision: {test_metrics['precision'] * 100:.2f}%")
print(f"Test Recall:    {test_metrics['recall'] * 100:.2f}%")
print(f"Test F1-Score:  {test_metrics['f1_score'] * 100:.2f}%")
print(f"Test ROC-AUC:   {test_roc_auc:.4f}")

print("\n==================== TEST CLASSIFICATION REPORT ====================")
print(classification_report(y_test, test_y_pred, target_names=["Benign", "Suspicious"]))

print("\n==================== PURE VALIDATION (DATASET 3 VAL) ====================")
print(f"Total pure benign samples: {len(pure_val_y_pred)}")
print(f"Specificity / Accuracy:    {pure_val_accuracy * 100:.2f}%")
print(f"False Positives:           {(pure_val_y_pred == 1).sum()} / {len(pure_val_y_pred)}")


==================== VALIDATION METRICS ====================
{
  "accuracy": 0.9787773933102653,
  "precision": 0.9579045837231057,
  "recall": 0.8469809760132341,
  "f1_score": 0.8990342405618964,
  "confusion_matrix": [
    [
      19167,
      90
    ],
    [
      370,
      2048
    ]
  ],
  "roc_auc": 0.8899434719429795
}

==================== UNSEEN TEST SET METRICS ====================
{
  "accuracy": 0.9797923875432526,
  "precision": 0.9647721935180836,
  "recall": 0.8498138187836161,
  "f1_score": 0.9036515618125824,
  "confusion_matrix": [
    [
      19183,
      75
    ],
    [
      363,
      2054
    ]
  ],
  "roc_auc": 0.8930831361079845
}
Test Accuracy:  97.98%
Test Precision: 96.48%
Test Recall:    84.98%
Test F1-Score:  90.37%
Test ROC-AUC:   0.8931

==================== TEST CLASSIFICATION REPORT ====================
              precision    recall  f1-score   support

      Benign       0.98      1.00      0.99     19258
  Suspicious       0.96      0.85      0

In [12]:
# ============================================================
# 11. Save Model and Feature Config
# ============================================================

model.save(MODEL_PATH)

feature_config = {
    "model_name": "IsolationForestScratch",
    "dataset_name": "Dataset 3",
    "feature_names": FEATURE_NAMES,
    "min_values": min_values.tolist(),
    "max_values": max_values.tolist(),
    "threshold": best_threshold,
    "initial_threshold_from_contamination": float(model.threshold_),
    "contamination": model.contamination,
    "n_trees": model.n_trees,
    "sample_size": model.sample_size,
    "model_file": str(MODEL_PATH),
    "validation_metrics": val_metrics,
    "test_metrics": test_metrics,
    "pure_val_accuracy": pure_val_accuracy,
}

with open(FEATURE_CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(feature_config, file, indent=2)

print("Saved model:", MODEL_PATH)
print("Saved feature config:", FEATURE_CONFIG_PATH)
print("\nFeature Config Summary:")
print(json.dumps({k: v for k, v in feature_config.items() if k not in ["min_values", "max_values"]}, indent=2))


Saved model: storage/isolation_forest.pkl
Saved feature config: storage/feature_config.json

Feature Config Summary:
{
  "model_name": "IsolationForestScratch",
  "dataset_name": "Dataset 3",
  "feature_names": [
    "url_length",
    "endpoint_depth",
    "query_param_count",
    "body_length",
    "header_count",
    "cookie_length",
    "user_agent_length",
    "special_char_count",
    "special_char_density",
    "sql_pattern_score",
    "xss_pattern_score",
    "traversal_pattern_score",
    "command_pattern_score",
    "log_injection_pattern_score",
    "log4j_pattern_score",
    "cookie_injection_pattern_score",
    "composite_anomaly_score",
    "response_body_length",
    "response_header_count",
    "status_code",
    "is_error_status"
  ],
  "threshold": 0.5752,
  "initial_threshold_from_contamination": 0.5752,
  "contamination": 0.1115,
  "n_trees": 300,
  "sample_size": 512,
  "model_file": "storage/isolation_forest.pkl",
  "validation_metrics": {
    "accuracy": 0.9787773

In [ ]:

loaded_model = IsolationForestScratch.load(MODEL_PATH)

with open(FEATURE_CONFIG_PATH, "r", encoding="utf-8") as file:
    loaded_config = json.load(file)

loaded_min_values = np.array(loaded_config["min_values"], dtype=float)
loaded_max_values = np.array(loaded_config["max_values"], dtype=float)
loaded_threshold = float(loaded_config["threshold"])

def predict_single_record(record: dict) -> dict:
    parser = RequestParser()
    extractor = FeatureExtractor()
    
    req = parser.normalize_record(record)
    features = extractor.extract(req)
    vector = extractor.to_vector(features)
    
    X_single = np.array([vector], dtype=float)
    denom = loaded_max_values - loaded_min_values
    denom[denom == 0] = 1.0
    X_scaled = (X_single - loaded_min_values) / denom
    
    score = float(loaded_model.anomaly_score(X_scaled)[0])
    is_anomaly = score >= loaded_threshold
    
    return {
        "endpoint": req.endpoint,
        "actual_attack_tag": req.attack_tag or "Benign",
        "anomaly_score": round(score, 4),
        "threshold": round(loaded_threshold, 4),
        "prediction": "Suspicious" if is_anomaly else "Benign",
        "is_anomaly": is_anomaly,
    }

# Test on 3 benign and 3 attack records
test_samples = [train_records[0], train_records[1], train_records[2]]
for r in train_records:
    if r.get("request", {}).get("Attack_Tag") and len(test_samples) < 6:
        test_samples.append(r)

print("\n--- Inference Verification on Saved Model ---")
for i, sample in enumerate(test_samples, 1):
    res = predict_single_record(sample)
    print(f"Sample {i}: {res}")



--- Inference Verification on Saved Model ---
Sample 1: {'endpoint': '/bookstore/signup', 'actual_attack_tag': 'Benign', 'anomaly_score': 0.5176, 'threshold': 0.5752, 'prediction': 'Benign', 'is_anomaly': False}
Sample 2: {'endpoint': '/forum', 'actual_attack_tag': 'XSS', 'anomaly_score': 0.6188, 'threshold': 0.5752, 'prediction': 'Suspicious', 'is_anomaly': True}
Sample 3: {'endpoint': '/about/greet', 'actual_attack_tag': 'Benign', 'anomaly_score': 0.3627, 'threshold': 0.5752, 'prediction': 'Benign', 'is_anomaly': False}
Sample 4: {'endpoint': '/forum', 'actual_attack_tag': 'XSS', 'anomaly_score': 0.6188, 'threshold': 0.5752, 'prediction': 'Suspicious', 'is_anomaly': True}
Sample 5: {'endpoint': '/categories/check/all', 'actual_attack_tag': 'LOG4J', 'anomaly_score': 0.6451, 'threshold': 0.5752, 'prediction': 'Suspicious', 'is_anomaly': True}
Sample 6: {'endpoint': '/forum', 'actual_attack_tag': 'XSS', 'anomaly_score': 0.6226, 'threshold': 0.5752, 'prediction': 'Suspicious', 'is_anoma